# Inventario y limpieza de imágenes (BCS vacas)

Esta libreta prepara y estandariza tus imágenes antes de realizar los recortes sin modificar nunca los archivos originales (todo se guarda en la carpeta nueva `clean/`). En un solo flujo, el código escanea tu carpeta en Drive para identificar cada foto con su respectiva vaca y tipo de vista, cruza estos hallazgos con tu archivo Excel para detectar cualquier discrepancia en los conteos, limpia y unifica los formatos convirtiendo todo a JPG real (corrigiendo la orientación, eliminando metadatos y duplicados), y finalmente genera el archivo `imagenes.csv` listo para el siguiente paso del proceso.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuración

In [ ]:
RUTA_RANCHO   = '/content/drive/MyDrive/lecheros/Dataset/Fotos Vacas Corral 6'
RUTA_EXTERNO  = '/content/drive/MyDrive/lecheros/Dataset/Dataset chino'
RUTA_EXCEL    = '/content/drive/MyDrive/lecheros/Dataset/EntendiendoDatos_folds.xlsx'
RUTA_LIMPIAS  = '/content/drive/MyDrive/lecheros/Dataset/clean'
RUTA_CSV      = '/content/drive/MyDrive/lecheros/Dataset/imagenes.csv'

CALIDAD_JPG   = 95
SOBRESCRIBIR  = False
UMBRAL_PARECIDAS = 5

## Instalación y Drive

In [ ]:
!pip -q install pillow-heif

if str(RUTA_RANCHO).startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import hashlib, re, unicodedata
from pathlib import Path

import pandas as pd
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener

register_heif_opener()
pd.set_option('display.max_rows', 200)

## Leer el Excel de vacas
De aquí salen el número de arete (`id_vaca`), el BCS y el fold de cada vaca.

In [ ]:
def normalizar(texto):
    """Minúsculas, sin acentos y sin guiones, para comparar nombres sin errores de dedo."""
    texto = unicodedata.normalize('NFKD', str(texto)).encode('ascii', 'ignore').decode()
    return re.sub(r'[\s_\-]+', ' ', texto.lower()).strip()

excel = pd.read_excel(RUTA_EXCEL)
excel = excel[excel['Vacas'].notna() & excel['NO Vac'].notna()].copy()

vacas = []
for _, fila in excel.iterrows():
    externa = str(fila['NO Vac']).strip().lower() == 'china'
    nombre = str(fila['Vacas']).strip()
    vacas.append({
        'id_vaca': f'ext_{normalizar(nombre)}' if externa else str(int(fila['NO Vac'])),
        'nombre': nombre,
        'nombre_pan': fila.get('Nombre (pan)', None),
        'bcs': float(fila['BCS']),
        'fold': 'train' if externa else fila.get('Fold', None),
        'fuente': 'externo' if externa else 'rancho',
        'total_excel': fila.get('Total', None),
        'lateral_excel': fila.get('Lateral', None),
    })
vacas = pd.DataFrame(vacas)

alias = {}
for _, v in vacas.iterrows():
    claves = [v['nombre'], v['id_vaca']]
    if pd.notna(v['nombre_pan']):
        claves.append(v['nombre_pan'])
    if v['fuente'] == 'externo':
        claves.append(f"{v['bcs']:g}")
    for c in claves:
        alias[normalizar(c)] = v['id_vaca']

info_vaca = vacas.set_index('id_vaca')
vacas[['id_vaca', 'nombre', 'nombre_pan', 'bcs', 'fold', 'fuente']]

##  Inventario de fotos

In [ ]:
EXTENSIONES = {'.jpg', '.jpeg', '.png', '.heic', '.heif'}

def formato_real(ruta):
    """Lee los primeros bytes para saber el formato verdadero, sin fiarse de la extensión."""
    with open(ruta, 'rb') as f:
        cabecera = f.read(16)
    if cabecera[:3] == b'\xff\xd8\xff':
        return 'jpg'
    if cabecera[:8] == b'\x89PNG\r\n\x1a\n':
        return 'png'
    if cabecera[4:8] == b'ftyp':
        return 'heic'
    return 'desconocido'

def detectar_vista(texto):
    t = normalizar(texto)
    palabras = set(t.split())
    if 'lateral' in palabras:
        return 'lateral'
    if 'picado' in palabras and ('sd' in palabras or 'semidorsal' in palabras):
        return 'sd_picado'
    if 'semidorsal' in palabras or 'sd' in palabras:
        return 'semidorsal'
    if 'picado' in palabras and 'dorsal' in palabras:
        return 'dorsal_picado'
    if 'dorsal' in palabras:
        return 'dorsal'
    return ''

def detectar_vaca(partes, nombre_archivo):
    for p in partes:
        if normalizar(p) in alias:
            return alias[normalizar(p)]
    base = re.sub(r'[_\-\s]*\d+$', '', Path(nombre_archivo).stem)
    for candidato in (base, base.split('_')[0]):
        if normalizar(candidato) in alias:
            return alias[normalizar(candidato)]
    return None

def inventario(raiz, fuente):
    filas = []
    if raiz is None or not Path(raiz).exists():
        print(f'(sin carpeta para {fuente}: {raiz})')
        return filas
    for ruta in sorted(Path(raiz).rglob('*')):
        if not ruta.is_file() or ruta.suffix.lower() not in EXTENSIONES:
            continue
        partes = ruta.relative_to(raiz).parts[:-1]
        vista = ''
        for texto in list(partes) + [ruta.stem]:
            vista = detectar_vista(texto) or vista
        fmt = formato_real(ruta)
        filas.append({
            'archivo_original': str(ruta),
            'fuente': fuente,
            'id_vaca': detectar_vaca(partes, ruta.name),
            'vista': vista,
            'formato_real': fmt,
            'extension_enganosa': fmt != 'desconocido' and fmt != ruta.suffix.lower().lstrip('.').replace('jpeg', 'jpg').replace('heif', 'heic'),
        })
    return filas

inv = pd.DataFrame(inventario(RUTA_RANCHO, 'rancho') + inventario(RUTA_EXTERNO, 'externo'))
print(f'Fotos encontradas: {len(inv)}')
print(inv['formato_real'].value_counts().to_string())
print(f"Archivos con extensión que no corresponde a su formato real: {int(inv['extension_enganosa'].sum())}")

### Revisión: lo que no cuadra


In [ ]:
sin_vaca = inv[inv['id_vaca'].isna()]
print(f'Fotos sin vaca reconocida: {len(sin_vaca)}')
if len(sin_vaca):
    display(sin_vaca[['archivo_original']].head(30))

sin_vista = inv[inv['id_vaca'].notna() & (inv['vista'] == '')]
print(f'Fotos sin vista detectada: {len(sin_vista)} (se pueden llenar después en el CSV)')

In [ ]:
encontradas = inv[inv['id_vaca'].notna()].copy()
encontradas['es_lateral'] = encontradas['vista'] == 'lateral'
conteo = (encontradas.groupby('id_vaca')['es_lateral']
          .agg(fotos_no_lateral=lambda s: int((~s).sum()), fotos_lateral='sum'))

revision = info_vaca[['nombre', 'nombre_pan', 'bcs', 'total_excel', 'lateral_excel']].join(conteo, how='left').fillna({'fotos_no_lateral': 0, 'fotos_lateral': 0})
revision['cuadra'] = (revision['fotos_no_lateral'] == revision['total_excel']) & (revision['fotos_lateral'] == revision['lateral_excel'])
revision.sort_values('cuadra')

## Limpieza

In [ ]:
def dhash(img, tam=8):
    """Huella perceptual: fotos casi iguales dan huellas parecidas."""
    g = img.convert('L').resize((tam + 1, tam), Image.LANCZOS)
    px = list(g.getdata())
    bits = [px[f * (tam + 1) + c] > px[f * (tam + 1) + c + 1] for f in range(tam) for c in range(tam)]
    return int(''.join('1' if b else '0' for b in bits), 2)

procesar = inv[inv['id_vaca'].notna()].copy()
procesar = procesar.sort_values(['fuente', 'id_vaca', 'vista', 'archivo_original']).reset_index(drop=True)

vistos, contador, resultados = {}, {}, []
for i, fila in procesar.iterrows():
    res = fila.to_dict()
    try:
        with Image.open(fila['archivo_original']) as img:
            img = ImageOps.exif_transpose(img)
            img = img.convert('RGB')
            huella = hashlib.md5(img.tobytes()).hexdigest()
            res.update(ancho=img.width, alto=img.height, hash=huella, dhash=dhash(img))

            if huella in vistos:
                res.update(estado='duplicado', duplicado_de=vistos[huella], archivo_limpio='')
                resultados.append(res)
                continue
            vistos[huella] = fila['archivo_original']

            vista = fila['vista'] or 'sinvista'
            clave = (fila['id_vaca'], vista)
            contador[clave] = contador.get(clave, 0) + 1
            destino = Path(RUTA_LIMPIAS) / fila['fuente'] / fila['id_vaca'] / f"{fila['id_vaca']}_{vista}_{contador[clave]:02d}.jpg"
            destino.parent.mkdir(parents=True, exist_ok=True)

            if SOBRESCRIBIR or not destino.exists():
                limpia = Image.frombytes('RGB', img.size, img.tobytes())
                limpia.save(destino, 'JPEG', quality=CALIDAD_JPG)
            res.update(estado='ok', duplicado_de='', archivo_limpio=str(destino))
    except Exception as e:
        res.update(estado=f'error: {e}', duplicado_de='', archivo_limpio='')
    resultados.append(res)
    if (i + 1) % 50 == 0:
        print(f'{i + 1}/{len(procesar)} procesadas')

limpias = pd.DataFrame(resultados)
print(limpias['estado'].value_counts().to_string())

###  Verificación


In [ ]:
con_metadatos = []
for ruta in limpias.loc[limpias['estado'] == 'ok', 'archivo_limpio']:
    with Image.open(ruta) as im:
        if len(im.getexif()) > 0 or 'xmp' in im.info or 'exif' in im.info:
            con_metadatos.append(ruta)
print(f'Fotos limpias con metadatos: {len(con_metadatos)}  (debe ser 0)')

ok = limpias[limpias['estado'] == 'ok'].reset_index(drop=True)
parecidas = []
for _, grupo in ok.groupby('id_vaca'):
    filas = grupo.to_dict('records')
    for a in range(len(filas)):
        for b in range(a + 1, len(filas)):
            d = bin(filas[a]['dhash'] ^ filas[b]['dhash']).count('1')
            if d <= UMBRAL_PARECIDAS:
                parecidas.append((filas[a]['archivo_limpio'], filas[b]['archivo_limpio'], d))
print(f'Pares casi iguales para revisar: {len(parecidas)}')
pd.DataFrame(parecidas, columns=['foto_a', 'foto_b', 'distancia']).head(30)

## 5. Guardar la tabla por imagen

In [ ]:
tabla = limpias.merge(vacas[['id_vaca', 'nombre', 'nombre_pan', 'bcs', 'fold']], on='id_vaca', how='left')
tabla['calidad'] = ''
tabla['usar'] = (tabla['estado'] == 'ok') & (tabla['vista'] != 'lateral')

columnas = ['archivo_limpio', 'id_vaca', 'nombre', 'nombre_pan', 'bcs', 'vista', 'fold', 'fuente',
            'calidad', 'usar', 'ancho', 'alto', 'formato_real', 'estado', 'duplicado_de', 'hash', 'archivo_original']
tabla = tabla[columnas]
tabla.to_csv(RUTA_CSV, index=False)
print(f'Guardado: {RUTA_CSV}  ({len(tabla)} filas)')


(tabla[tabla['usar']].groupby(['fuente', 'id_vaca', 'nombre_pan', 'bcs', 'fold'], dropna=False)
 .size().rename('fotos').reset_index().sort_values(['fuente', 'bcs']))